# Confidence ablation

Tables and charts for one finished sweep of `scripts/run_ablation.py`.

```bash
python scripts/run_ablation.py --device 0            # run the sweep
python scripts/run_ablation.py --device 0 --resume   # continue an interrupted one
```

Confidence is the only factor that changes: same dev sequences, counting rules, seed and tracker
settings. At confidence 0.1 the tracker also sees detections below its high threshold (0.25); from
0.25 up they are removed before tracking. Latency is detector + tracker time per frame, without
image reading and report writing.

Set `ABLATION_DIR` to read another sweep (default `outputs/ablation/confidence_dev`).

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# The repository root is the nearest parent that holds src/
ROOT = next(d for d in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (d / "src").is_dir())
sys.path.insert(0, str(ROOT))
from src.utils import file_info, read_json

ABLATION_DIR = (ROOT / os.environ.get("ABLATION_DIR", "outputs/ablation/confidence_dev")).resolve()
OUTPUT_DIR = ABLATION_DIR / "analysis"
CLASSES = ["car", "bus", "truck", "all"]
plt.style.use("seaborn-v0_8-whitegrid")

## Check the sweep

The tables are built only from a complete sweep whose files still match the hashes in the manifest,
so a partial or edited experiment cannot pass as a full comparison.

In [ ]:
manifest = read_json(ABLATION_DIR / "manifest.json")
if not manifest.get("complete"):
    raise ValueError("The sweep is not finished: run scripts/run_ablation.py with --resume")
for key, outputs in manifest["stages"].items():
    for path, sha256 in outputs.items():
        if file_info(ABLATION_DIR / path)["sha256"] != sha256:
            raise ValueError(f"Output changed since the sweep: {path} (stage {key})")

plan = manifest["plan"]
sequences, confidences, jobs = plan["sequences"], plan["confidences"], plan["jobs"]
print(f"{len(jobs)} runs, {len(sequences)} dev sequences, confidences {confidences}")
print(f"Environment: {manifest['tracking_environment']}")

## Tables

One row per confidence and configuration.
MAE: mean absolute count error per sequence, in vehicles.
WAPE: sum of absolute errors / sum of GT counts, in percent (n/a when the class has no GT vehicles).
SCE: predicted minus GT count, so negative means undercounting.
HOTA, MOTA and IDF1 are the dataset-level values TrackEval computes from all sequences together.

In [ ]:
LABELS = {"bytetrack": "ByteTrack", "botsort": "BoT-SORT"}
summary_rows, class_rows, sequence_rows = [], [], []

for confidence in confidences:
    reports = ABLATION_DIR / f"conf{confidence:g}/reports"
    counting = read_json(reports / "counting_eval_dev.json")
    tracking = read_json(reports / "tracking_eval_dev.json")

    for job in (j for j in jobs if j["confidence"] == confidence):
        config = job["config"]
        timing = read_json(ABLATION_DIR / f"conf{confidence:g}/tracks/dev/{config}/timing.json")
        approach = f"{Path(job['model']).stem.replace('yolov8', 'YOLOv8')} + {LABELS[job['tracker']]}"
        key = {"Approach": approach, "Config": config, "Confidence": confidence}

        for cls in CLASSES:
            r = counting[config][cls]       # pred, gt and sce hold one number per sequence, in plan order
            class_rows.append({**key, "Class": cls, "GT total": sum(r["gt"]), "Pred total": sum(r["pred"]),
                               "MAE": r["mae"], "WAPE (%)": None if r["wape"] is None else 100 * r["wape"],
                               "Mean SCE": r["mean_sce"]})
            sequence_rows += [{**key, "Class": cls, "Sequence": seq, "GT": gt, "Pred": pred, "SCE": pred - gt,
                               "Absolute error": abs(pred - gt)}
                              for seq, gt, pred in zip(sequences, r["gt"], r["pred"], strict=True)]

        # The last class row is "all": the summary adds the tracking and latency columns to it
        combined = tracking[config]["aggregate"]["combined"]
        summary_rows.append({**class_rows[-1], "HOTA (%)": combined["HOTA"], "MOTA (%)": combined["MOTA"],
                             "IDF1 (%)": combined["IDF1"], "IDSW": tracking[config]["aggregate"]["sum"]["IDSW"],
                             "Detector ms": timing["det_ms_per_frame"], "Tracker ms": timing["trk_ms_per_frame"],
                             "Latency ms": timing["total_ms_per_frame"], "FPS": timing["fps"]})

summary, per_class, per_sequence = map(pd.DataFrame, (summary_rows, class_rows, sequence_rows))
report_table = summary[["Confidence", "Approach", "MAE", "WAPE (%)", "IDF1 (%)", "Latency ms"]]
print(report_table.round(3).to_markdown(index=False))

## Charts

Compare points on one curve to see the effect of confidence on one pipeline, and curves at the same
confidence to compare detectors and trackers. The zero line in the SCE chart separates overcounting
from undercounting. The heatmap shows whether one busy sequence explains a change.

In [ ]:
CHARTS_DIR, TABLES_DIR = OUTPUT_DIR / "charts", OUTPUT_DIR / "tables"
CHARTS_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)


def plot_curves(axis, column, title, zero=False):
    """One curve per detector + tracker pair, confidence on the x axis."""
    for approach, part in summary.groupby("Approach", sort=False):
        axis.plot(part["Confidence"], part[column], marker="o", label=approach)
    axis.set(title=title, xlabel="Tracker high-score threshold (detector runs at 0.1)", ylabel=column, xticks=confidences)
    if zero:
        axis.axhline(0, color="black", linewidth=1)
    axis.legend(fontsize=8)


def save_figure(figure, name):
    """Save as PNG and as vector PDF (for the report), then show it."""
    figure.tight_layout()
    for extension in ("png", "pdf"):
        figure.savefig(CHARTS_DIR / f"{name}_dev.{extension}", dpi=200, bbox_inches="tight")
    plt.show()
    plt.close(figure)


# Two panels per figure; the y axis starts at 0 because these values cannot be negative
PANELS = {"counting_sensitivity": [("Counting error", "WAPE (%)"), ("Error in vehicles", "MAE")],
          "tracking_sensitivity": [("Identity quality", "IDF1 (%)"), ("Identity switches", "IDSW")],
          "latency_sensitivity": [("Detector time", "Detector ms"), ("Tracker time", "Tracker ms")]}
for name, panels in PANELS.items():
    figure, axes = plt.subplots(1, 2, figsize=(13, 5))
    for axis, (title, column) in zip(axes, panels, strict=True):
        plot_curves(axis, column, title)
        axis.set_ylim(bottom=0)
    save_figure(figure, name)

figure, axis = plt.subplots(figsize=(9, 5))
plot_curves(axis, "Mean SCE", "Direction of the counting error", zero=True)
save_figure(figure, "signed_counting_error")

# Signed error per sequence and condition, centred on 0 so over- and undercounting get opposite colours
errors = per_sequence[per_sequence["Class"] == "all"].assign(
    Condition=lambda d: d["Approach"] + " conf " + d["Confidence"].map("{:g}".format))
grid = errors.pivot(index="Condition", columns="Sequence", values="SCE")
grid = grid.reindex(index=errors["Condition"].drop_duplicates(), columns=sequences)
limit = max(1, float(np.abs(grid.to_numpy()).max()))
figure, axis = plt.subplots(figsize=(11, max(4, 0.4 * len(grid))))
sns.heatmap(grid, annot=True, fmt=".0f", cmap="RdBu_r", center=0, vmin=-limit, vmax=limit, ax=axis,
            cbar_kws={"label": "Predicted minus GT count"})
axis.set_title("Counting error per dev sequence")
save_figure(figure, "sequence_errors")

## Export

CSV keeps full precision. Markdown and LaTeX use three decimals and show undefined values as n/a.

Questions for the write-up:
- Does a lower confidence help every sequence or only some?
- Do better identity metrics also reduce the counting error?
- Does the detector or the tracker change the shape of the curve?
- Does a small total error hide a large error in one class?

In [ ]:
for name, table in (("ablation_summary", summary), ("ablation_report", report_table),
                    ("ablation_per_class", per_class), ("ablation_per_sequence", per_sequence)):
    table.to_csv(TABLES_DIR / f"{name}_dev.csv", index=False)

    # Text tables: floats with three decimals, integers unchanged
    text = table.astype(object)
    for column in table.select_dtypes("floating"):
        text[column] = table[column].map(lambda v: "n/a" if pd.isna(v) else f"{v:.3f}")
    (TABLES_DIR / f"{name}_dev.md").write_text(text.to_markdown(index=False, disable_numparse=True) + "\n",
                                               encoding="utf-8")
    (TABLES_DIR / f"{name}_dev.tex").write_text(text.to_latex(index=False, escape=True), encoding="utf-8")
print(f"Saved tables and charts to {OUTPUT_DIR}")